# Lab 2 — Precision Recruiting with Recommender Systems

**Mission:** Limited recruiter time means we cannot visit every school. Build a transparent system that answers:

1. **WHERE should we focus?** Rank schools by downstream value and operational feasibility.
2. **WHAT should we try there?** Compare collaborative and content-based evidence to recommend an engagement.

You will deliberately begin with the wrong objective, watch the ranking change, and then infer a promising action Jefferson High has never tried.

**Estimated time:** 75 minutes (Lab A: 30; Lab B: 45)

> **Use your coding assistant as a teammate.** Give it the current cell, the self-check output, and the goal. Ask it to explain the smallest useful change rather than rewriting the notebook.

Suggested prompt:

> I am working in a classroom Jupyter notebook. Explain what this self-check is testing, then suggest the smallest edit to the marked variables. Do not change the data or the test.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(128, 128, 128, 0.12); border: 1px solid rgba(128, 128, 128, 0.28); border-radius: 6px; padding: 12px 16px; margin: 8px 0 12px;">\n  <h2 style="margin: 0 0 8px;">0. Setup</h2>\n  <p style="margin: 0 0 14px;">Run these setup blocks before beginning Lab A.</p>\n  <h3 style="margin: 0 0 6px;">0.1 Define the notebook self-check helpers</h3>\n  <p style="margin: 0;">This block defines the reusable <code>check()</code> and <code>mission_header()</code> helpers used throughout the lab.</p>\n</div>'))

0. Setup 
 Run these setup blocks before beginning Lab A. 
 0.1 Define the notebook self-check helpers 
 This block defines the reusable check() and mission_header() helpers used throughout the lab.

In [ ]:
from IPython.display import display, Markdown

def check(name, condition, hint=""):
    try:
        passed = bool(condition)
    except Exception as exc:
        passed = False
        hint = f"{hint} ({type(exc).__name__}: {exc})"
    icon = "✅" if passed else "❌"
    print(f"{icon} {name}")
    if not passed and hint:
        print(f"   Hint: {hint}")
    return passed

def mission_header(text):
    display(Markdown(f"> **Mission checkpoint:** {text}"))

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(128, 128, 128, 0.12); border: 1px solid rgba(128, 128, 128, 0.28); border-radius: 6px; padding: 12px 16px; margin: 8px 0 12px;">\n  <h3 style="margin: 0 0 6px;">0.2 Import packages and configure the notebook</h3>\n  <p style="margin: 0;">This block loads the Python packages used for data work, charts, scaling, and similarity, then sets reproducible display options.</p>\n</div>'))

0.2 Import packages and configure the notebook 
 This block loads the Python packages used for data work, charts, scaling, and similarity, then sets reproducible display options.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
pd.set_option("display.max_columns", 30)
pd.options.display.float_format = "{:,.3f}".format

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(128, 128, 128, 0.12); border: 1px solid rgba(128, 128, 128, 0.28); border-radius: 6px; padding: 12px 16px; margin: 8px 0 12px;">\n  <h3 style="margin: 0 0 6px;">0.3 Read the prepared Lab 1 artifacts and Lab 2 profile tables</h3>\n  <p>Lab 2 uses validated CSV artifacts produced by the Lab 1 pipeline. Prepared copies are supplied so this lab remains runnable even if a team has not completed Lab 1.</p>\n  <p>It also uses two small content-profile tables created for this lab:</p>\n  <ul>\n    <li><code>school_profiles.csv</code> describes aggregate program emphasis at each school.</li>\n    <li><code>action_profiles.csv</code> describes the six actions already present in the historical event data along the same axes.</li>\n  </ul>\n  <p style="margin-bottom: 0;">All records and profile scores are fictional classroom data; protected characteristics are not used.</p>\n</div>'))

0.3 Read the prepared Lab 1 artifacts and Lab 2 profile tables 
 Lab 2 uses validated CSV artifacts produced by the Lab 1 pipeline. Prepared copies are supplied so this lab remains runnable even if a team has not completed Lab 1. 
 It also uses two small content-profile tables created for this lab: 
 
 school_profiles.csv describes aggregate program emphasis at each school. 
 action_profiles.csv describes the six actions already present in the historical event data along the same axes. 
 
 All records and profile scores are fictional classroom data; protected characteristics are not used.

In [ ]:
DATA_URL = "https://usard-demo.netlify.app/data"

schools = pd.read_csv(f"{DATA_URL}/school_summary.csv")
engagements = pd.read_csv(f"{DATA_URL}/clean_recruiting_events.csv", parse_dates=["event_date"])
school_profiles = pd.read_csv(f"{DATA_URL}/school_profiles.csv").set_index("school_name")
action_profiles = pd.read_csv(f"{DATA_URL}/action_profiles.csv").set_index("action")

print(
    f"Loaded {len(schools)} school summaries, {len(engagements)} clean events, "
    f"{len(school_profiles)} school profiles, and {len(action_profiles)} action profiles."
)
schools.head(8)

# Lab A — WHERE should we focus?

## A1. Rank by activity

If appointments are the goal, Lincoln appears to win. Pause before running the next cell: is appointment volume the outcome USARD actually wants to optimize?

In [ ]:
appointment_ranking = schools.nlargest(5, "appointments")[["school_name", "appointments", "qualified", "contracts"]]
appointment_ranking

## A2. Look downstream

Complete the four marked column choices. The cell runs even before it is correct; the checks tell you what to fix.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background: linear-gradient(135deg, rgba(245, 158, 11, 0.18), rgba(59, 130, 246, 0.10)); border: 1px solid rgba(245, 158, 11, 0.55); border-left: 6px solid #f59e0b; border-radius: 8px; padding: 14px 16px; margin: 10px 0 12px;">\n  <p style="font-size: 1.05em; margin: 0 0 10px;"><strong>🛠️ TODO:</strong> Fill out the numerator and denominator columns for qualification rate and efficiency.</p>\n  <p style="margin: 0 0 10px;"><strong>🤔 Think:</strong> If we are trying to measure qualification rate and downstream efficiency, which table columns should we compare?</p>\n  <div style="background-color: rgba(59, 130, 246, 0.12); border-left: 4px solid #3b82f6; border-radius: 4px; padding: 9px 11px;">\n    <strong>💡 Hint:</strong> Look at the columns in the table above. A rate compares an outcome with the opportunity that produced it; efficiency compares a downstream outcome with the constrained resource used.\n  </div>\n</div>'))

🛠️ TODO: Fill out the numerator and denominator columns for qualification rate and efficiency. 
 🤔 Think: If we are trying to measure qualification rate and downstream efficiency, which table columns should we compare? 
 
 💡 Hint: Look at the columns in the table above. A rate compares an outcome with the opportunity that produced it; efficiency compares a downstream outcome with the constrained resource used.

In [ ]:
QUALIFIED_NUMERATOR = "appointments"   # TODO
QUALIFIED_DENOMINATOR = "appointments" # TODO
EFFICIENCY_NUMERATOR = "appointments"  # TODO
EFFICIENCY_DENOMINATOR = "recruiter_hours"

schools["qualified_rate"] = (
    schools[QUALIFIED_NUMERATOR] / schools[QUALIFIED_DENOMINATOR]
)
schools["contracts_per_hour"] = (
    schools[EFFICIENCY_NUMERATOR] / schools[EFFICIENCY_DENOMINATOR]
)

schools.nlargest(5, "contracts_per_hour")[[
    "school_name", "appointments", "qualified", "contracts", "contracts_per_hour"
]]

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;">\n  <p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only expand and run the cell directly below once you are ready to check your work. <strong>⬇️</strong></p>\n</div>'))

⬇️ 🔒 Checking cell below: Only expand and run the cell directly below once you are ready to check your work. ⬇️

In [ ]:
#@title
check("Qualification rate uses qualified ÷ appointments",
      QUALIFIED_NUMERATOR == "qualified" and QUALIFIED_DENOMINATOR == "appointments",
      "The numerator is the number that made it through qualification.")
check("Efficiency uses contracts ÷ recruiter hours",
      EFFICIENCY_NUMERATOR == "contracts" and EFFICIENCY_DENOMINATOR == "recruiter_hours",
      "We care about downstream success per constrained hour.")
check("Lincoln's appointment volume does not make it the efficiency leader",
      schools.nlargest(1, "appointments").iloc[0]["school_name"] != schools.nlargest(1, "contracts_per_hour").iloc[0]["school_name"],
      "Fix the efficiency numerator first.")

## A3. Define an explainable opportunity score

The mission owner chose:

- 60% downstream efficiency
- 25% qualification rate
- 15% school access

Edit the three weights. The algorithm cannot decide the objective for us.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background: linear-gradient(135deg, rgba(245, 158, 11, 0.18), rgba(59, 130, 246, 0.10)); border: 1px solid rgba(245, 158, 11, 0.55); border-left: 6px solid #f59e0b; border-radius: 8px; padding: 14px 16px; margin: 10px 0 12px;">\n  <p style="font-size: 1.05em; margin: 0 0 10px;"><strong>🛠️ TODO:</strong> Fill out the three opportunity-score weights.</p>\n  <p style="margin: 0 0 10px;"><strong>🤔 Think:</strong> How should the full 100% be divided among downstream efficiency, qualification rate, and school access?</p>\n  <div style="background-color: rgba(59, 130, 246, 0.12); border-left: 4px solid #3b82f6; border-radius: 4px; padding: 9px 11px;">\n    <strong>💡 Hint:</strong> Translate the stated 60%, 25%, and 15% priorities into decimals. The three weights should add up to 1.\n  </div>\n</div>'))

🛠️ TODO: Fill out the three opportunity-score weights. 
 🤔 Think: How should the full 100% be divided among downstream efficiency, qualification rate, and school access? 
 
 💡 Hint: Translate the stated 60%, 25%, and 15% priorities into decimals. The three weights should add up to 1.

In [ ]:
scaler = MinMaxScaler()
features = ["contracts_per_hour", "qualified_rate", "access_score"]
schools[["success_norm", "qualified_norm", "access_norm"]] = scaler.fit_transform(schools[features])

SUCCESS_WEIGHT = .34   # TODO
QUALIFIED_WEIGHT = .33 # TODO
ACCESS_WEIGHT = .33    # TODO

schools["opportunity_score"] = (
    SUCCESS_WEIGHT * schools["success_norm"]
    + QUALIFIED_WEIGHT * schools["qualified_norm"]
    + ACCESS_WEIGHT * schools["access_norm"]
)

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;">\n  <p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only expand and run the cell directly below once you are ready to check your work. <strong>⬇️</strong></p>\n</div>'))

⬇️ 🔒 Checking cell below: Only expand and run the cell directly below once you are ready to check your work. ⬇️

In [ ]:
#@title
check("Weights sum to 1", np.isclose(SUCCESS_WEIGHT + QUALIFIED_WEIGHT + ACCESS_WEIGHT, 1.0))
check("Weights match the mission objective",
      np.allclose([SUCCESS_WEIGHT, QUALIFIED_WEIGHT, ACCESS_WEIGHT], [.60, .25, .15]),
      "Translate 60%, 25%, and 15% into decimals.")

## A4. Filter infeasible or thinly supported options

Scores do not override operations. Set the thresholds to **30 miles** and at least **4 historical events**. Event count is observable evidence volume—not a made-up “data quality” score.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background: linear-gradient(135deg, rgba(245, 158, 11, 0.18), rgba(59, 130, 246, 0.10)); border: 1px solid rgba(245, 158, 11, 0.55); border-left: 6px solid #f59e0b; border-radius: 8px; padding: 14px 16px; margin: 10px 0 12px;">\n  <p style="font-size: 1.05em; margin: 0 0 10px;"><strong>🛠️ TODO:</strong> Fill out the maximum-distance and minimum-history thresholds.</p>\n  <p style="margin: 0 0 10px;"><strong>🤔 Think:</strong> What limits would keep the shortlist operationally feasible and supported by enough historical evidence?</p>\n  <div style="background-color: rgba(59, 130, 246, 0.12); border-left: 4px solid #3b82f6; border-radius: 4px; padding: 9px 11px;">\n    <strong>💡 Hint:</strong> Use the two limits stated directly above: 30 miles and at least 4 historical events.\n  </div>\n</div>'))

🛠️ TODO: Fill out the maximum-distance and minimum-history thresholds. 
 🤔 Think: What limits would keep the shortlist operationally feasible and supported by enough historical evidence? 
 
 💡 Hint: Use the two limits stated directly above: 30 miles and at least 4 historical events.

In [ ]:
MAX_DISTANCE = 999       # TODO
MIN_HISTORICAL_EVENTS = 0 # TODO

eligible = schools.loc[
    schools["distance_miles"].le(MAX_DISTANCE)
    & schools["historical_events"].ge(MIN_HISTORICAL_EVENTS)
].copy()

excluded = schools.loc[~schools.index.isin(eligible.index), [
    "school_name", "distance_miles", "historical_events", "opportunity_score"
]].sort_values("opportunity_score", ascending=False)
display(Markdown("**Excluded options**"))
display(excluded)

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;">\n  <p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only expand and run the cell directly below once you are ready to check your work. <strong>⬇️</strong></p>\n</div>'))

⬇️ 🔒 Checking cell below: Only expand and run the cell directly below once you are ready to check your work. ⬇️

In [ ]:
#@title
check("Distance threshold is operationally correct", MAX_DISTANCE == 30)
check("Evidence threshold is operationally correct", MIN_HISTORICAL_EVENTS == 4)
check("Liberty is excluded for travel", "Liberty High" not in set(eligible["school_name"]))
check("Victory is excluded for insufficient history", "Victory High" not in set(eligible["school_name"]))
check("No arbitrary data-quality metric is used", "data_quality" not in schools.columns)

## A5. Return Top K

Set `K = 5`. The winning school at the top of this list becomes the target for Lab B.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background: linear-gradient(135deg, rgba(245, 158, 11, 0.18), rgba(59, 130, 246, 0.10)); border: 1px solid rgba(245, 158, 11, 0.55); border-left: 6px solid #f59e0b; border-radius: 8px; padding: 14px 16px; margin: 10px 0 12px;">\n  <p style="font-size: 1.05em; margin: 0 0 10px;"><strong>🛠️ TODO:</strong> Fill out the number of schools to return in the final shortlist.</p>\n  <p style="margin: 0 0 10px;"><strong>🤔 Think:</strong> How many feasible schools should the recruiter receive before choosing the top-ranked target?</p>\n  <div style="background-color: rgba(59, 130, 246, 0.12); border-left: 4px solid #3b82f6; border-radius: 4px; padding: 9px 11px;">\n    <strong>💡 Hint:</strong> The requested shortlist size is shown immediately above. Set <code>K</code> to that number.\n  </div>\n</div>'))

🛠️ TODO: Fill out the number of schools to return in the final shortlist. 
 🤔 Think: How many feasible schools should the recruiter receive before choosing the top-ranked target? 
 
 💡 Hint: The requested shortlist size is shown immediately above. Set K to that number.

In [ ]:
K = 3  # TODO
top_schools = eligible.nlargest(K, "opportunity_score").copy()
top_schools[["school_name", "opportunity_score", "contracts_per_hour", "qualified_rate", "access_score"]]

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;">\n  <p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only expand and run the cell directly below once you are ready to check your work. <strong>⬇️</strong></p>\n</div>'))

⬇️ 🔒 Checking cell below: Only expand and run the cell directly below once you are ready to check your work. ⬇️

In [ ]:
#@title
check("Top K returns five schools", K == 5 and len(top_schools) == 5)
check("Jefferson wins the final school ranking", top_schools.iloc[0]["school_name"] == "Jefferson High")

ax = top_schools.sort_values("opportunity_score").plot.barh(
    x="school_name", y="opportunity_score", legend=False, color="#1f5a91", figsize=(8, 4)
)
ax.set(title="Recommended schools after scoring and constraints", xlabel="Opportunity score", ylabel="")
plt.tight_layout()
plt.show()

> **Aha:** A recommender can produce a perfectly correct ranking for the wrong objective. There is no universal “best” school—only a ranking tied to an objective, evidence, and constraints.

# Lab B — WHAT should we do there?

Lab A selected Jefferson High. We will make the same recommendation two ways and then compare them:

1. **Collaborative Filtering:** What worked at schools that behaved like Jefferson?
2. **Content-Based Filtering:** Which actions fit Jefferson's program profile?

Keeping the evidence streams separate makes the final hybrid score explainable.

## Collaborative Filtering

Treat schools like “users,” engagement actions like “items,” and historical contracts per recruiter-hour like a “rating.” Collaborative filtering uses outcome patterns; it does not need the school or action profile tables.

In [ ]:
actions = action_profiles.index.tolist()

print(f"The event artifact contains {len(engagements):,} validated engagements.")
engagements.sample(8, random_state=SEED)[[
    "engagement_id", "event_date", "school_name", "action",
    "recruiter_hours", "contracts"
]]

### B1. Build the school × action matrix

What does the blank cell for Jefferson + Mechanical mean? It means **unobserved**, not failed.

In [ ]:
school_action_summary = (
    engagements
    .groupby(["school_name", "action"])
    .agg(
        total_hours=("recruiter_hours", "sum"),
        total_contracts=("contracts", "sum"),
        event_count=("engagement_id", "count"),
    )
)
school_action_summary["effectiveness"] = (
    school_action_summary["total_contracts"]
    / school_action_summary["total_hours"]
)

school_action = (
    school_action_summary["effectiveness"]
    .unstack()
    .reindex(columns=actions)
)
event_counts = (
    school_action_summary["event_count"]
    .unstack()
    .reindex(columns=actions)
)

display(Markdown("**Contracts per recruiter-hour**"))
display(school_action.style.format("{:.2f}", na_rep="—").background_gradient(cmap="Blues", axis=None))
display(Markdown("**Historical event count behind each score**"))
display(event_counts.style.format("{:.0f}", na_rep="—"))

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;">\n  <p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only expand and run the cell directly below once you are ready to check your work. <strong>⬇️</strong></p>\n</div>'))

⬇️ 🔒 Checking cell below: Only expand and run the cell directly below once you are ready to check your work. ⬇️

In [ ]:
#@title
TARGET_SCHOOL = top_schools.iloc[0]["school_name"]
check("Lab B follows Lab A's winning school", TARGET_SCHOOL == "Jefferson High")
check("Jefferson has no Mechanical history", pd.isna(school_action.loc[TARGET_SCHOOL, "Mechanical Careers Demo"]))
check("Jefferson does have Healthcare history", pd.notna(school_action.loc[TARGET_SCHOOL, "Healthcare Careers Session"]))
check("Missing does not become zero", not (school_action.fillna(-1).loc[TARGET_SCHOOL, "Mechanical Careers Demo"] == 0))

### B2. Find behaviorally similar schools

Cosine similarity should use only actions observed at both schools. Require at least **three** overlapping actions so a one- or two-action coincidence cannot dominate the neighborhood.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background: linear-gradient(135deg, rgba(245, 158, 11, 0.18), rgba(59, 130, 246, 0.10)); border: 1px solid rgba(245, 158, 11, 0.55); border-left: 6px solid #f59e0b; border-radius: 8px; padding: 14px 16px; margin: 10px 0 12px;">\n  <p style="font-size: 1.05em; margin: 0 0 10px;"><strong>🛠️ TODO:</strong> Fill out the minimum number of overlapping actions required for similarity.</p>\n  <p style="margin: 0 0 10px;"><strong>🤔 Think:</strong> How much shared behavioral evidence should two schools have before we trust their similarity score?</p>\n  <div style="background-color: rgba(59, 130, 246, 0.12); border-left: 4px solid #3b82f6; border-radius: 4px; padding: 9px 11px;">\n    <strong>💡 Hint:</strong> The evidence rule above requires at least three actions observed at both schools.\n  </div>\n</div>'))

🛠️ TODO: Fill out the minimum number of overlapping actions required for similarity. 
 🤔 Think: How much shared behavioral evidence should two schools have before we trust their similarity score? 
 
 💡 Hint: The evidence rule above requires at least three actions observed at both schools.

In [ ]:
MIN_OVERLAP = 1  # TODO

def cosine_on_overlap(a, b, min_overlap=MIN_OVERLAP):
    mask = a.notna() & b.notna()
    if mask.sum() < min_overlap:
        return np.nan
    x = a[mask].to_numpy(dtype=float)
    y = b[mask].to_numpy(dtype=float)
    denominator = np.linalg.norm(x) * np.linalg.norm(y)
    return np.nan if denominator == 0 else float(np.dot(x, y) / denominator)

target_vector = school_action.loc[TARGET_SCHOOL]
overlap_counts = pd.Series({
    school: int((target_vector.notna() & school_action.loc[school].notna()).sum())
    for school in school_action.index if school != TARGET_SCHOOL
}, name="overlap_count")
similarities = pd.Series({
    school: cosine_on_overlap(target_vector, school_action.loc[school])
    for school in school_action.index if school != TARGET_SCHOOL
}, name="similarity").dropna().sort_values(ascending=False)

similarity_table = pd.concat([similarities, overlap_counts], axis=1).dropna().sort_values("similarity", ascending=False)
similarity_table

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;">\n  <p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only expand and run the cell directly below once you are ready to check your work. <strong>⬇️</strong></p>\n</div>'))

⬇️ 🔒 Checking cell below: Only expand and run the cell directly below once you are ready to check your work. ⬇️

In [ ]:
#@title
check("Similarity requires at least three overlaps", MIN_OVERLAP == 3,
      "One or two shared actions are too little evidence for a stable neighborhood.")
check("Washington is Jefferson's closest behavioral neighbor", similarities.index[0] == "Washington High")
check("The neighborhood is meaningfully spread out", similarities.median() < .90,
      "Require three overlaps and verify that the school profiles are not all pointing in nearly the same direction.")

### B3. Predict an untried action

Turn on similarity weighting. A close neighbor should contribute more than a weak neighbor. To avoid diluting the signal with every weakly related school, use the three nearest behavioral neighbors.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background: linear-gradient(135deg, rgba(245, 158, 11, 0.18), rgba(59, 130, 246, 0.10)); border: 1px solid rgba(245, 158, 11, 0.55); border-left: 6px solid #f59e0b; border-radius: 8px; padding: 14px 16px; margin: 10px 0 12px;">\n  <p style="font-size: 1.05em; margin: 0 0 10px;"><strong>🛠️ TODO:</strong> Turn on similarity weighting for the neighbor prediction.</p>\n  <p style="margin: 0 0 10px;"><strong>🤔 Think:</strong> Should a very similar school influence the prediction more than a weakly similar school?</p>\n  <div style="background-color: rgba(59, 130, 246, 0.12); border-left: 4px solid #3b82f6; border-radius: 4px; padding: 9px 11px;">\n    <strong>💡 Hint:</strong> Change the Boolean control so the weighted-average branch runs. The neighbor count is already set for you.\n  </div>\n</div>'))

🛠️ TODO: Turn on similarity weighting for the neighbor prediction. 
 🤔 Think: Should a very similar school influence the prediction more than a weakly similar school? 
 
 💡 Hint: Change the Boolean control so the weighted-average branch runs. The neighbor count is already set for you.

In [ ]:
USE_SIMILARITY_WEIGHTS = False  # TODO
NEIGHBOR_COUNT = 3

def predict_action(action, matrix=school_action, sims=similarities):
    evidence = []
    for school, similarity in sims.head(NEIGHBOR_COUNT).items():
        value = matrix.loc[school, action]
        if pd.notna(value) and similarity > 0:
            evidence.append((school, float(similarity), float(value)))
    if not evidence:
        return np.nan
    if USE_SIMILARITY_WEIGHTS:
        numerator = sum(sim * value for _, sim, value in evidence)
        denominator = sum(sim for _, sim, _ in evidence)
        return numerator / denominator
    return np.mean([value for _, _, value in evidence])

mechanical_prediction = predict_action("Mechanical Careers Demo")
print(f"Predicted contracts per recruiter-hour: {mechanical_prediction:.3f}")

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;">\n  <p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only expand and run the cell directly below once you are ready to check your work. <strong>⬇️</strong></p>\n</div>'))

⬇️ 🔒 Checking cell below: Only expand and run the cell directly below once you are ready to check your work. ⬇️

In [ ]:
#@title
check("Prediction uses similarity weighting", USE_SIMILARITY_WEIGHTS)
check("Mechanical is predicted to be promising", mechanical_prediction > .60,
      "Verify the similarity-weighted average and the target's missing cell.")

### B4. Rank observed and predicted actions together

Keep provenance visible. A predicted score is not the same kind of evidence as an observed score.

In [ ]:
recommendations = school_action.loc[TARGET_SCHOOL].copy()
evidence_type = pd.Series("observed", index=recommendations.index)

for action in recommendations.index[recommendations.isna()]:
    recommendations[action] = predict_action(action)
    evidence_type[action] = "predicted"

action_ranking = pd.DataFrame({
    "score": recommendations,
    "evidence": evidence_type,
}).sort_values("score", ascending=False)

action_ranking.head(3)

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;">\n  <p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only expand and run the cell directly below once you are ready to check your work. <strong>⬇️</strong></p>\n</div>'))

⬇️ 🔒 Checking cell below: Only expand and run the cell directly below once you are ready to check your work. ⬇️

In [ ]:
#@title
check("Mechanical is the top recommendation", action_ranking.index[0] == "Mechanical Careers Demo")
check("Mechanical is labeled predicted", action_ranking.loc["Mechanical Careers Demo", "evidence"] == "predicted")
originally_observed = school_action.loc[TARGET_SCHOOL].dropna().index
originally_missing = school_action.loc[TARGET_SCHOOL].index[school_action.loc[TARGET_SCHOOL].isna()]
check("Observed actions remain labeled observed", (evidence_type.loc[originally_observed] == "observed").all())
check("Every filled blank is labeled predicted", (evidence_type.loc[originally_missing] == "predicted").all())

## Content-Based Filtering

Content-based filtering uses descriptive features rather than other schools' outcomes. The school and action tables share five 0–1 axes: cyber, engineering, mechanical, healthcare, and education.

The action table contains exactly the six action types already found in the historical event data. A blank in the school × action matrix therefore remains an untried historical action—not a brand-new catalog item.

In [ ]:
dimensions = ["cyber", "engineering", "mechanical", "healthcare", "education"]

display(Markdown(f"**{TARGET_SCHOOL} program profile**"))
display(school_profiles.loc[[TARGET_SCHOOL], dimensions])
display(Markdown("**Action profiles on the same axes**"))
display(action_profiles.loc[actions, dimensions])

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;">\n  <p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only expand and run the cell directly below once you are ready to check your work. <strong>⬇️</strong></p>\n</div>'))

⬇️ 🔒 Checking cell below: Only expand and run the cell directly below once you are ready to check your work. ⬇️

In [ ]:
#@title
check("Every historical action has one content profile",
      set(engagements["action"].unique()) == set(action_profiles.index))
check("Every ranked school has one content profile",
      set(schools["school_name"]).issubset(set(school_profiles.index)))
check("School profile values stay on the 0–1 scale",
      school_profiles[dimensions].ge(0).all().all() and school_profiles[dimensions].le(1).all().all())
check("Action profile values stay on the 0–1 scale",
      action_profiles[dimensions].ge(0).all().all() and action_profiles[dimensions].le(1).all().all())

### B5. Score school–action fit

Cosine similarity now compares Jefferson's program profile with each action profile. This is the same similarity measure as before, but the vectors represent content features rather than historical outcomes.

In [ ]:
target_content_vector = school_profiles.loc[TARGET_SCHOOL, dimensions].to_numpy().reshape(1, -1)
action_content_matrix = action_profiles.loc[actions, dimensions].to_numpy()

content_scores = pd.Series(
    cosine_similarity(target_content_vector, action_content_matrix)[0],
    index=actions,
    name="content_score",
)
content_ranking = content_scores.sort_values(ascending=False).to_frame()
content_ranking

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;">\n  <p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only expand and run the cell directly below once you are ready to check your work. <strong>⬇️</strong></p>\n</div>'))

⬇️ 🔒 Checking cell below: Only expand and run the cell directly below once you are ready to check your work. ⬇️

In [ ]:
#@title
check("Content filtering scores every available action", content_scores.notna().all())
check("Content scores stay between zero and one", content_scores.between(0, 1).all())
check("Mechanical has strong content fit for Jefferson",
      content_scores["Mechanical Careers Demo"] > .75)

### B6. Build an explainable hybrid ranking

Normalize the collaborative scores, then give them 60% of the final score and content fit 40%. Collaborative evidence receives more weight because it is tied directly to historical outcomes; content evidence helps explain program fit.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background: linear-gradient(135deg, rgba(245, 158, 11, 0.18), rgba(59, 130, 246, 0.10)); border: 1px solid rgba(245, 158, 11, 0.55); border-left: 6px solid #f59e0b; border-radius: 8px; padding: 14px 16px; margin: 10px 0 12px;">\n  <p style="font-size: 1.05em; margin: 0 0 10px;"><strong>🛠️ TODO:</strong> Fill out the collaborative and content weights for the hybrid score.</p>\n  <p style="margin: 0 0 10px;"><strong>🤔 Think:</strong> How much influence should historical outcome evidence have compared with descriptive content fit?</p>\n  <div style="background-color: rgba(59, 130, 246, 0.12); border-left: 4px solid #3b82f6; border-radius: 4px; padding: 9px 11px;">\n    <strong>💡 Hint:</strong> Convert the stated 60% collaborative and 40% content policy into decimals. The weights must add up to 1.\n  </div>\n</div>'))

🛠️ TODO: Fill out the collaborative and content weights for the hybrid score. 
 🤔 Think: How much influence should historical outcome evidence have compared with descriptive content fit? 
 
 💡 Hint: Convert the stated 60% collaborative and 40% content policy into decimals. The weights must add up to 1.

In [ ]:
COLLABORATIVE_WEIGHT = .50  # TODO
CONTENT_WEIGHT = .50        # TODO

collaborative_norm = (
    (recommendations - recommendations.min())
    / (recommendations.max() - recommendations.min())
)
hybrid_ranking = pd.DataFrame({
    "collaborative_score": recommendations,
    "collaborative_norm": collaborative_norm,
    "content_score": content_scores,
    "collaborative_evidence": evidence_type,
})
hybrid_ranking["hybrid_score"] = (
    COLLABORATIVE_WEIGHT * hybrid_ranking["collaborative_norm"]
    + CONTENT_WEIGHT * hybrid_ranking["content_score"]
)
hybrid_ranking = hybrid_ranking.sort_values("hybrid_score", ascending=False)
hybrid_ranking

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;">\n  <p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only expand and run the cell directly below once you are ready to check your work. <strong>⬇️</strong></p>\n</div>'))

⬇️ 🔒 Checking cell below: Only expand and run the cell directly below once you are ready to check your work. ⬇️

In [ ]:
#@title
check("Hybrid weights sum to 1",
      np.isclose(COLLABORATIVE_WEIGHT + CONTENT_WEIGHT, 1.0))
check("Hybrid weights match the stated evidence policy",
      np.allclose([COLLABORATIVE_WEIGHT, CONTENT_WEIGHT], [.60, .40]),
      "Translate 60% collaborative and 40% content into decimals.")
check("Mechanical remains the top hybrid recommendation",
      hybrid_ranking.index[0] == "Mechanical Careers Demo")
check("The hybrid table preserves collaborative provenance",
      hybrid_ranking.loc["Mechanical Careers Demo", "collaborative_evidence"] == "predicted")

## Red-team pause

Discuss before deployment:

- Are we learning what works—or what recruiters historically chose to try?
- Who assigned the content-profile scores, and how often should they be refreshed?
- How old can evidence be before it becomes stale?
- Should a prediction based on two overlapping actions receive the same confidence as one based on five?
- Which fields must never be used as ranking features?

**Transition:** We now have a WHERE and a WHAT. The recruiter still needs authoritative information before acting. That is the job of RAG.